# 🩺 Digital Twin for Type 2 Diabetes — Notebook 2: Teaching the Twin to Predict Spikes

**Goal:** every 15 minutes, answer one question for each patient:

> **"Will this patient's glucose go above 180 mg/dL in the next 2 hours?"**

The model gets to use **only the past** (what has already happened up to that moment) — exactly like it would in real life.

**Plan of this notebook**
1. Load our three data files
2. Clean the CGM data (real sensors have gaps)
3. Create the "answer key" (label): did a spike actually happen in the next 2 hours?
4. Create **features** — the clues the model uses (recent sugar trend, meals, walking, last night's sleep, the patient's hospital record, and the patient's *personal* history)
5. Train the model and test it on patients it has **never seen**
6. Prove that **combining EHR + wearables beats either one alone** (this is the competition's core requirement)
7. Measure results the way a doctor cares about: *how many spikes did we catch, how early, and how many false alarms?*

## Step 1 — Load the data
In Colab, click the 📁 folder icon on the left and drag in the three CSV files
(`ehr_patients.csv`, `wearable_timeseries.csv`, `daily_summary.csv`). If you forget, the cell below will ask you to upload them.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FILES = ["ehr_patients.csv", "wearable_timeseries.csv", "daily_summary.csv"]
missing = [f for f in FILES if not os.path.exists(f)]
if missing:
    from google.colab import files
    print("Please upload:", missing)
    files.upload()

ehr = pd.read_csv("ehr_patients.csv")
wear = pd.read_csv("wearable_timeseries.csv", parse_dates=["timestamp"])
daily = pd.read_csv("daily_summary.csv", parse_dates=["date"])
print(ehr.shape, wear.shape, daily.shape)

## Step 2 — Clean the glucose signal

Real CGMs (like Abbott FreeStyle Libre or Dexcom) can only display values between 40 and 400 mg/dL, and they drop readings now and then.
- We **cap** values to the 40–400 range.
- We **fill short gaps** (up to 30 minutes) by drawing a straight line between the neighbouring readings — standard practice.
- **Longer gaps stay empty**: we don't invent data we don't have. Predictions are skipped when the recent data is missing.

In [ ]:
wear = wear.sort_values(["patient_id", "timestamp"]).reset_index(drop=True)
wear["cgm"] = wear.cgm_glucose_mg_dl.clip(40, 400)
wear["cgm"] = wear.groupby("patient_id").cgm.transform(
    lambda s: s.interpolate(limit=6, limit_area="inside"))   # 6 readings x 5 min = 30 min
print(f"Missing before cleaning: {wear.cgm_glucose_mg_dl.isna().mean():.2%}  |  after: {wear.cgm.isna().mean():.2%}")

## Step 3 — The answer key: was there a spike in the next 2 hours?

For every moment, we look **forward** 2 hours (24 readings) and check whether glucose went above 180.

⚠️ An important design choice: we only make predictions when the patient is **currently below 180**.
If sugar is *already* high, "predicting" a spike is pointless — the doctor can already see it. The real value is warning **before** it happens.
(Many student projects miss this and report inflated accuracy. Judges will notice that we didn't.)

In [ ]:
HORIZON = 24          # 24 x 5 min = 2 hours
SPIKE = 180

def future_max(s, n):
    # highest value in the NEXT n readings (not including now)
    return s[::-1].rolling(n, min_periods=int(n * 0.5)).max()[::-1].shift(-1)

wear["future_max_2h"] = wear.groupby("patient_id").cgm.transform(lambda s: future_max(s, HORIZON))
wear["spike_next_2h"] = (wear.future_max_2h > SPIKE).astype(float)
wear.loc[wear.future_max_2h.isna(), "spike_next_2h"] = np.nan

## Step 4 — Features: the clues the model uses

Think of how an experienced diabetes doctor would guess whether a spike is coming. They'd ask:

| Clue group | Questions the doctor asks | Data source |
|---|---|---|
| **Glucose trend** | Where is sugar now? Is it rising, and how fast? | Wearable (CGM) |
| **Food** | Did they just eat? How many carbs? How long ago? | Wearable (meal log) |
| **Activity** | Did they walk after eating? | Wearable (steps) |
| **Heart** | Is heart rate elevated? | Wearable (watch) |
| **Sleep & recovery** | Did they sleep badly last night? Low HRV? | Wearable (daily summary) |
| **Time** | Is it after lunch? Early morning (dawn effect)? Weekend? | Clock |
| **Who they are** | HbA1c, BMI, medication, genetics, years with diabetes | **EHR** |
| **Personal history ("the twin")** | How much does *this* person's sugar usually rise after a meal? What's their usual level? | Learned from their own past data |

The last group is what makes this a **digital twin** rather than a generic model: the twin keeps learning each individual's
personal response, using **only data from before the current moment**.

In [ ]:
def add_patient_features(d):
    d = d.copy()
    g = d.cgm
    # --- glucose trend ---
    for lag in [3, 6, 12]:                        # 15, 30, 60 min ago
        d[f"cgm_lag_{lag * 5}m"] = g.shift(lag)
    d["cgm_slope_15m"] = (g - g.shift(3)) / 15    # mg/dL per minute
    d["cgm_slope_30m"] = (g - g.shift(6)) / 30
    d["cgm_mean_2h"] = g.rolling(24, min_periods=12).mean()
    d["cgm_std_2h"] = g.rolling(24, min_periods=12).std()
    d["cgm_max_2h"] = g.rolling(24, min_periods=12).max()

    # --- food ---
    c = d.carbs_logged_g
    d["carbs_last_30m"] = c.rolling(6, min_periods=1).sum()
    d["carbs_last_1h"] = c.rolling(12, min_periods=1).sum()
    d["carbs_last_2h"] = c.rolling(24, min_periods=1).sum()
    d["carbs_last_4h"] = c.rolling(48, min_periods=1).sum()
    meal_idx = pd.Series(np.where(c > 0, np.arange(len(d)), np.nan), index=d.index).ffill()
    d["minutes_since_meal"] = ((np.arange(len(d)) - meal_idx) * 5).clip(upper=720).fillna(720)

    # --- activity & heart ---
    d["steps_last_30m"] = d.steps.rolling(6, min_periods=1).sum()
    d["steps_last_1h"] = d.steps.rolling(12, min_periods=1).sum()
    d["hr_now"] = d.heart_rate_bpm
    d["hr_mean_30m"] = d.heart_rate_bpm.rolling(6, min_periods=1).mean()
    d["asleep_now"] = (~d.sleep_stage.isin(["Not asleep"])).astype(int)

    # --- time ---
    hour = d.timestamp.dt.hour + d.timestamp.dt.minute / 60
    d["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    d["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    d["is_weekend"] = (d.timestamp.dt.dayofweek >= 5).astype(int)

    # --- personal history (the twin learns this patient) ---
    d["personal_mean_glucose"] = g.expanding(min_periods=12).mean().shift(1)
    d["personal_pct_above_180"] = (g > SPIKE).astype(float).expanding(min_periods=12).mean().shift(1)

    # Personal meal response: how much did glucose rise after THIS patient's past meals?
    rises = np.full(len(d), np.nan)
    per_gram = np.full(len(d), np.nan)
    history, history_pg = [], []
    meal_positions = np.where(c.values > 0)[0]
    gv = g.values
    next_meal = 0
    completed = []   # (time the meal's 2h window closes, rise, rise per gram)
    for m in meal_positions:
        window = gv[m:m + 25]
        if np.isnan(gv[m]) or np.all(np.isnan(window)):
            continue
        rise = np.nanmax(window) - gv[m]
        completed.append((m + 24, rise, rise / c.values[m]))
    completed.sort()
    j = 0
    for t in range(len(d)):
        while j < len(completed) and completed[j][0] < t:
            history.append(completed[j][1]); history_pg.append(completed[j][2]); j += 1
        if history:
            rises[t] = np.mean(history[-10:])      # average of last 10 meals
            per_gram[t] = np.mean(history_pg[-10:])
    d["personal_meal_rise"] = rises
    d["personal_rise_per_gram"] = per_gram
    d["expected_rise_now"] = d.personal_rise_per_gram * d.carbs_last_2h   # twin's own forecast
    return d

wear = pd.concat([add_patient_features(d) for _, d in wear.groupby("patient_id")], ignore_index=True)

# --- sleep & recovery: last night's sleep is recorded on the morning's date ---
daily["prev_day_steps"] = daily.groupby("patient_id").total_steps.shift(1)
daily["hrv_vs_personal_avg"] = daily.overnight_hrv_rmssd_ms / daily.groupby("patient_id") \
    .overnight_hrv_rmssd_ms.transform(lambda s: s.expanding().mean().shift(1))
wear["date"] = wear.timestamp.dt.normalize()
wear = wear.merge(daily[["patient_id", "date", "sleep_hours", "sleep_efficiency_pct", "deep_sleep_pct",
                         "overnight_hrv_rmssd_ms", "hrv_vs_personal_avg", "prev_day_steps"]],
                  on=["patient_id", "date"], how="left")
# ⚠️ We deliberately do NOT use today's total steps — at 2 pm, the watch can't know tonight's steps yet (that would be "cheating" = data leakage).

# --- EHR (static) features ---
ehr_f = ehr.copy()
ehr_f["sex_male"] = (ehr_f.sex == "Male").astype(int)
ehr_f["on_second_drug"] = ehr_f.medication.str.contains("DPP-4|SGLT2").astype(int)
ehr_f["on_insulin"] = ehr_f.medication.str.contains("insulin").astype(int)
ehr_f["on_metformin"] = ehr_f.medication.str.contains("Metformin").astype(int)
ehr_f["has_hypertension"] = ehr_f.past_diagnoses.str.contains("Hypertension").astype(int)
ehr_f["has_dyslipidemia"] = ehr_f.past_diagnoses.str.contains("Dyslipidemia").astype(int)
ehr_f["family_history"] = ehr_f.family_history_diabetes.astype(int)
ehr_f["diet_rice"] = (ehr_f.diet_pattern == "Rice-dominant").astype(int)
ehr_f["activity_score"] = ehr_f.activity_level.map({"Low": 0, "Moderate": 1, "High": 2})
EHR_FEATURES = ["age", "sex_male", "bmi", "years_with_diabetes", "hba1c_pct", "fasting_glucose_mg_dl",
                "tcf7l2_risk_alleles", "family_history", "on_metformin", "on_second_drug", "on_insulin",
                "systolic_bp", "ldl_mg_dl", "hdl_mg_dl", "triglycerides_mg_dl", "creatinine_mg_dl",
                "has_hypertension", "has_dyslipidemia", "diet_rice", "activity_score"]
wear = wear.merge(ehr_f[["patient_id"] + EHR_FEATURES], on="patient_id", how="left")

WEARABLE_FEATURES = ["cgm", "cgm_lag_15m", "cgm_lag_30m", "cgm_lag_60m", "cgm_slope_15m", "cgm_slope_30m",
                     "cgm_mean_2h", "cgm_std_2h", "cgm_max_2h",
                     "carbs_last_30m", "carbs_last_1h", "carbs_last_2h", "carbs_last_4h", "minutes_since_meal",
                     "steps_last_30m", "steps_last_1h", "hr_now", "hr_mean_30m", "asleep_now",
                     "sleep_hours", "sleep_efficiency_pct", "deep_sleep_pct", "overnight_hrv_rmssd_ms",
                     "hrv_vs_personal_avg", "prev_day_steps", "hour_sin", "hour_cos", "is_weekend"]
TWIN_FEATURES = ["personal_mean_glucose", "personal_pct_above_180", "personal_meal_rise",
                 "personal_rise_per_gram", "expected_rise_now"]
ALL_FEATURES = WEARABLE_FEATURES + TWIN_FEATURES + EHR_FEATURES
print(f"{len(ALL_FEATURES)} features: {len(WEARABLE_FEATURES)} wearable, {len(TWIN_FEATURES)} personal-twin, {len(EHR_FEATURES)} EHR")

### Build the prediction table
We make one prediction **every 15 minutes**, only when: current glucose is known and below 180, and there's enough future data to know the answer.
We also skip each patient's first 2 hours (no history yet).

In [ ]:
rows = wear[(wear.timestamp.dt.minute % 15 == 0) & wear.cgm.notna() & (wear.cgm < SPIKE)
            & wear.spike_next_2h.notna() & wear.cgm_lag_60m.notna()].copy()
print(f"Prediction moments: {len(rows):,}")
print(f"Share that led to a spike within 2h: {rows.spike_next_2h.mean():.1%}")

## Step 5 — Fair testing: split by patient

If we trained and tested on the *same* patients, the model could just memorise them. That's not how it'll be used —
a hospital will onboard **new** patients. So we split **by patient**:

- **60 patients → training** (the model learns from them)
- **10 patients → validation** (we tune the alarm setting on them)
- **30 patients → testing** (locked away until the very end — the honest exam)

In [ ]:
pids = np.array(sorted(ehr.patient_id))
rng = np.random.default_rng(7)
rng.shuffle(pids)
train_ids, val_ids, test_ids = pids[:60], pids[60:70], pids[70:]
train, val, test = [rows[rows.patient_id.isin(ids)] for ids in (train_ids, val_ids, test_ids)]
print(len(train), len(val), len(test))

## Step 6 — Train the model (LightGBM)

**LightGBM** builds hundreds of small decision trees ("if glucose is rising fast AND carbs > 50g in the last hour AND no walk, then risk is high…").
Each tree fixes the mistakes of the previous ones. It's fast, accurate on tables of numbers, handles missing values on its own,
and is widely used in real hospital risk models.

We train **four versions** to answer the key question — *does fusing EHR + wearables help?*
1. **Simple rule** (no AI): "warn if glucose is above 150 and rising"
2. **EHR only**: just the hospital record
3. **Wearables only**: sensors, no hospital record
4. **Digital Twin (fused)**: wearables + EHR + personal history

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score

PARAMS = dict(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=50,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1, random_state=42)

def fit(features):
    m = lgb.LGBMClassifier(**PARAMS)
    m.fit(train[features], train.spike_next_2h)
    return m

models = {"EHR only": EHR_FEATURES,
          "Wearables only": WEARABLE_FEATURES,
          "Digital Twin (EHR + wearables + personal)": ALL_FEATURES}
fitted, results = {}, []

rule = ((test.cgm > 150) & (test.cgm_slope_15m > 0)).astype(float)
results.append(dict(model="Simple rule (no AI)", AUROC=roc_auc_score(test.spike_next_2h, rule),
                    AUPRC=average_precision_score(test.spike_next_2h, rule)))
for name, feats in models.items():
    m = fit(feats)
    fitted[name] = (m, feats)
    p = m.predict_proba(test[feats])[:, 1]
    results.append(dict(model=name, AUROC=roc_auc_score(test.spike_next_2h, p),
                        AUPRC=average_precision_score(test.spike_next_2h, p)))

results = pd.DataFrame(results).round(3)
print(results.to_string(index=False))

### How to read these scores
- **AUROC** (0.5 = coin toss, 1.0 = perfect): if you pick one moment that led to a spike and one that didn't, how often does the model rank the spike one as riskier?
- **AUPRC**: similar, but focuses on how *precise* the warnings are. For comparison, random guessing would score about the spike rate printed above.

If the fused Digital Twin scores highest, we've proven the competition's main claim with evidence. That table is a key slide.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(results.model, results.AUROC, color=["grey", "#9bb", "#6aa", "#07a"])
ax.set_xlim(0.5, 1.0); ax.set_xlabel("AUROC on 30 unseen patients (higher is better)")
for i, v in enumerate(results.AUROC):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center")
ax.set_title("Fusing EHR + wearables + personal history gives the best predictions")
plt.tight_layout(); plt.show()

## Step 7 — Results a doctor cares about

Doctors don't think in AUROC. They ask three things:
1. **Sensitivity** — of all real spikes, how many did we warn about *before* they happened?
2. **Lead time** — how many minutes of warning did the patient get? (More time = time to take a walk or adjust food.)
3. **False alarms per day** — too many false alarms and people ignore the app (called **alarm fatigue**, a well-known problem in hospitals).

The model outputs a risk from 0–100%. We choose the alarm level (threshold) on the **validation** patients, so the test stays honest.

In [ ]:
twin_model, twin_feats = fitted["Digital Twin (EHR + wearables + personal)"]

def spike_events(full):
    """A clinically meaningful spike: glucose goes above 180, STAYS there for at least 15 minutes,
    and was below 180 for the previous 30 minutes (so sensor wobble around 180 isn't counted many times)."""
    above = (full.cgm.rolling(3, min_periods=1).mean() > SPIKE)
    sustained = above & above.shift(-1, fill_value=False) & above.shift(-2, fill_value=False)
    was_below = ~above.shift(1, fill_value=False).rolling(6, min_periods=1).max().astype(bool)
    return full.index[sustained & was_below]

def alarm_episodes(alarm_times):
    """Consecutive alarms (every 15 min) count as ONE notification to the patient/doctor."""
    episodes, start, prev = [], None, None
    for t in alarm_times:
        if prev is None or t - prev > pd.Timedelta(minutes=15):
            if start is not None:
                episodes.append((start, prev))
            start = t
        prev = t
    if start is not None:
        episodes.append((start, prev))
    return episodes

def evaluate_alarms(df, threshold):
    caught, lead_times, false_alarms, total_alarms, patient_days, total_spikes = 0, [], 0, 0, 0, 0
    for pid, d in df.groupby("patient_id"):
        full = wear[wear.patient_id == pid].set_index("timestamp")
        patient_days += full.index.normalize().nunique()
        spikes = spike_events(full)
        alarms = d[d.risk >= threshold].timestamp.sort_values()
        for c in spikes:
            total_spikes += 1
            before = alarms[(alarms < c) & (alarms >= c - pd.Timedelta(hours=2))]
            if len(before):
                caught += 1
                lead_times.append((c - before.min()).total_seconds() / 60)
        for start, end in alarm_episodes(list(alarms)):
            total_alarms += 1
            if not ((spikes >= start) & (spikes <= end + pd.Timedelta(hours=2))).any():
                false_alarms += 1
    return dict(threshold=threshold, spikes=total_spikes,
                sensitivity=caught / max(total_spikes, 1),
                median_lead_min=float(np.median(lead_times)) if lead_times else 0.0,
                alarms_per_day=total_alarms / patient_days,
                false_alarms_per_day=false_alarms / patient_days)

val = val.assign(risk=twin_model.predict_proba(val[twin_feats])[:, 1])
test = test.assign(risk=twin_model.predict_proba(test[twin_feats])[:, 1])

sweep = pd.DataFrame([evaluate_alarms(val, t) for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]]).round(2)
print("Validation patients — trade-off between catching spikes and false alarms:")
print(sweep.to_string(index=False))

# Pick the lowest threshold (= catch the most spikes) that keeps false alarms at or under 1 per day
ok = sweep[sweep.false_alarms_per_day <= 1.0]
THRESHOLD = float(ok.threshold.min()) if len(ok) else 0.5
print(f"\nChosen alarm threshold: {THRESHOLD:.0%} risk")

final = evaluate_alarms(test, THRESHOLD)
print(f"\n🎯 FINAL RESULT on 30 never-seen patients:")
print(f"   Spikes caught in advance: {final['sensitivity']:.0%} of {final['spikes']} spikes")
print(f"   Alarms raised:            {final['alarms_per_day']:.1f} per patient per day")
print(f"   Median warning time:      {final['median_lead_min']:.0f} minutes before the spike")
print(f"   False alarms:             {final['false_alarms_per_day']:.1f} per patient per day")

## Step 8 — See the twin in action
The top panel shows glucose with the moments the twin raised an alarm (red triangles). The bottom panel shows the twin's risk score over time.
Look for alarms appearing **before** the line crosses 180.

In [ ]:
def show_patient(pid, day=5):
    d = wear[wear.patient_id == pid]
    d = d[d.timestamp.dt.normalize() == d.timestamp.dt.normalize().unique()[day]]
    r = test[(test.patient_id == pid) & (test.timestamp.dt.normalize() == d.timestamp.iloc[0].normalize())]
    fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    ax[0].plot(d.timestamp, d.cgm, lw=1.5, label="Glucose")
    ax[0].axhline(SPIKE, c="red", ls="--", lw=1)
    for t, c in d[d.carbs_logged_g > 0][["timestamp", "carbs_logged_g"]].values:
        ax[0].axvline(t, c="orange", alpha=0.5)
        ax[0].text(t, 0.03, f" {c}g", color="darkorange", fontsize=9, transform=ax[0].get_xaxis_transform())
    al = r[r.risk >= THRESHOLD]
    ax[0].scatter(al.timestamp, al.cgm, marker="^", c="red", s=40, zorder=3, label="Twin alarm")
    ax[0].set_ylabel("mg/dL"); ax[0].legend(loc="upper left")
    ax[0].set_title(f"Patient {pid} (unseen during training)")
    grid = pd.date_range(d.timestamp.iloc[0], d.timestamp.iloc[-1], freq="15min")
    rr = r.set_index("timestamp").risk.reindex(grid)          # gaps = already above 180, no prediction needed
    ax[1].fill_between(grid, rr * 100, alpha=0.4, color="purple", label="Risk (blank = already above 180)")
    ax[1].legend(loc="upper left", fontsize=8)
    ax[1].axhline(THRESHOLD * 100, c="purple", ls="--", lw=1)
    ax[1].set_ylabel("Spike risk (%)"); ax[1].set_ylim(0, 100)
    plt.tight_layout(); plt.show()

show_patient(test_ids[0])

## Step 9 — Save the trained twin
We save the model so the doctor dashboard (next notebook) can load it. Download `twin_model.txt` and `test_patients.csv` and keep them with your data files.

In [ ]:
twin_model.booster_.save_model("twin_model.txt")
pd.Series(test_ids, name="patient_id").to_csv("test_patients.csv", index=False)
results.to_csv("model_comparison.csv", index=False)
try:
    from google.colab import files
    for f in ["twin_model.txt", "test_patients.csv", "model_comparison.csv"]:
        files.download(f)
except ImportError:
    print("Saved locally.")